# Identificação de M2 com Gaia DR3

Neste exemplo usamos o AI-Scope para buscar estrelas ao redor de M2 e identificar sua população a partir de uma sobredensidade no espaço de movimentos próprios.

A posição do aglomerado é conhecida, mas seu movimento próprio não é fornecido ao método.

### Gaia — Gaia DR3
O **Gaia** é um telescópio espacial da ESA dedicado à astrometria da Via Láctea.

- **Instrumento:** satélite com telescópios ópticos, fotometria e espectroscopia.
- **Dados:** posições, paralaxes, movimentos próprios, magnitudes e informações espectroscópicas.
- **Gaia DR3:** terceira grande publicação de dados, com aproximadamente 1,8 bilhão de fontes.


- **Gaia DR3:** https://www.cosmos.esa.int/web/gaia/dr3


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import adss

cl = adss.ADSSClient(
    base_url="https://ai-scope.cbpf.br/",
)


## Consulta ao Gaia DR3

Em `public.gaia_dr3`, selecione até **50.000** fontes dentro de `RADIUS_DEG` de (`CENTER_RA`, `CENTER_DEC`). Retorne ID, coordenadas, paralaxe e erro, movimentos próprios e erros, fotometria G/BP/RP, `bp_rp`, `ruwe` e a distância angular `radius_deg` calculada por `dist`. Exija movimentos próprios e magnitudes G/BP/RP presentes e `phot_g_mean_mag < 19`. `cone` faz o corte espacial no servidor; `dist` devolve graus.


In [43]:
metadata = cl.get_database_metadata()

table = metadata.get_table("public", "gaia_dr3")
if table is not None:
    print(table.column_names()[:30])

['solution_id', 'designation', 'source_id', 'random_index', 'ref_epoch', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax', 'parallax_error', 'parallax_over_error', 'pm', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 'ra_dec_corr', 'ra_parallax_corr', 'ra_pmra_corr', 'ra_pmdec_corr', 'dec_parallax_corr', 'dec_pmra_corr', 'dec_pmdec_corr', 'parallax_pmra_corr', 'parallax_pmdec_corr', 'pmra_pmdec_corr', 'astrometric_n_obs_al', 'astrometric_n_obs_ac', 'astrometric_n_good_obs_al']


In [ ]:
CENTER_RA = 323.3626
CENTER_DEC = -0.8233
RADIUS_DEG = 1.0

query = None # Complete a consulta AstroQL do enunciado.
if query is None:
    raise NotImplementedError("Complete query;")

gaia = cl.query_and_wait(query_text=query, mode="astroql").data

print(f"{len(gaia):,} fontes")
gaia.head()


## Seleção de qualidade


In [ ]:
clean = gaia[
    (gaia.ruwe < 1.4) &
    (gaia.pmra_error < 1.0) &
    (gaia.pmdec_error < 1.0)
].copy()

print(f"{len(clean):,} fontes após os cortes")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(clean.pmra, clean.pmdec, s=3, alpha=0.12)
ax.set_xlim(-20, 20)
ax.set_ylim(-20, 15)
ax.set_xlabel(r"$mu_alpha$ [mas/ano]")
ax.set_ylabel(r"$mu_delta$ [mas/ano]")
ax.set_title("Movimentos próprios")
plt.show()

## Região central e campo de controle

A região central contém uma contribuição maior do aglomerado. Uma região externa é usada para estimar a distribuição cinemática das estrelas de campo.


In [ ]:
core = clean[clean.radius_deg < 0.05].copy()

field = clean[
    (clean.radius_deg >= 0.5) &
    (clean.radius_deg <= 1.0)
].copy()

print(f"Centro: {len(core):,}")
print(f"Campo:  {len(field):,}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(field.pmra, field.pmdec, s=3, alpha=0.08, label="Campo")
ax.scatter(core.pmra, core.pmdec, s=10, alpha=0.6, label="Centro")

ax.set_xlim(-15, 15)
ax.set_ylim(-15, 10)
ax.set_xlabel(r"$\mu_{\alpha*}$ [mas/ano]")
ax.set_ylabel(r"$\mu_{\delta}$ [mas/ano]")
ax.legend()
plt.show()


## Sobredensidade cinemática — KDE

Estrelas de um mesmo aglomerado tendem a apresentar movimentos próprios semelhantes. Para identificar possíveis membros, comparamos a distribuição cinemática da região central com a de uma região de campo.

Utilizamos o **Kernel Density Estimation (KDE)**, que estima a densidade de estrelas no espaço de movimentos próprios ($\mu_{\alpha*}$, $\mu_\delta$), suavizando a distribuição dos pontos.

Calculamos duas distribuições:
- **Core:** densidade cinemática das estrelas próximas ao centro.
- **Field:** densidade cinemática das estrelas de campo.

A sobredensidade cinemática é calculada por:

$$
E = \log_{10}\left(\frac{\rho_{\mathrm{core}}+\epsilon}{\rho_{\mathrm{field}}+\epsilon}\right)
$$

- **$E > 0$:** maior densidade relativa no centro.
- **$E \approx 0$:** densidades semelhantes.
- **$E < 0$:** maior densidade relativa no campo.

Selecionamos os **2,5% de estrelas com maior excesso cinemático** como candidatas a membros do aglomerado.

Esse valor representa um contraste entre distribuições, não uma probabilidade de associação.

In [ ]:
pm_min, pm_max = -20, 20
pd_min, pd_max = -20, 15

core_pm = core[
    core.pmra.between(pm_min, pm_max) &
    core.pmdec.between(pd_min, pd_max)
]

field_pm = field[
    field.pmra.between(pm_min, pm_max) &
    field.pmdec.between(pd_min, pd_max)
]

kde_core = gaussian_kde(np.vstack([core_pm.pmra, core_pm.pmdec]))
kde_field = gaussian_kde(np.vstack([field_pm.pmra, field_pm.pmdec]))

xy = np.vstack([clean.pmra, clean.pmdec])

p_core = kde_core(xy)
p_field = kde_field(xy)

# O valor de epsilon é definido como o percentil 1% das densidades positivas do campo.
floor = np.percentile(p_field[p_field > 0], 1)

clean["kinematic_excess"] = np.log10(
    (p_core + floor) / (p_field + floor)
)


## Candidatos a membros


In [ ]:
threshold = clean.kinematic_excess.quantile(0.975)

cluster = clean[
    clean.kinematic_excess >= threshold
].copy()

print(f"Limite: {threshold:.3f}")
print(f"Candidatos: {len(cluster):,}")

### Distribuição espacial


In [ ]:
from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(clean.ra, clean.dec, s=1, alpha=0.04, label="Campo")
ax.scatter(cluster.ra, cluster.dec, s=2, alpha=0.6, label="Candidatos")
ax.scatter(CENTER_RA, CENTER_DEC, marker="+", s=150, color="black", label="Centro", zorder=11)

r = 0.05
ax.add_patch(Ellipse(
    (CENTER_RA, CENTER_DEC),
    width=2*r / np.cos(np.deg2rad(CENTER_DEC)),
    height=2*r,
    fill=False, edgecolor="blue", linewidth=1.5,
    linestyle="--", zorder=10, label="Core (0.05°)"
))

ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_aspect(1 / np.cos(np.deg2rad(CENTER_DEC)))
ax.invert_xaxis()
ax.legend()

plt.show()

### Diagrama cor-magnitude

No diagrama cor–magnitude, `bp_rp` é a cor (diferença entre magnitudes Gaia BP e RP). O eixo de G é invertido porque objetos mais brilhantes têm magnitude menor.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    clean.bp_rp,
    clean.phot_g_mean_mag,
    s=3,
    alpha=0.03,
    label="Campo"
)

ax.scatter(
    cluster.bp_rp,
    cluster.phot_g_mean_mag,
    s=9,
    alpha=0.6,
    label="Candidatos"
)

ax.set_xlim(-0.5, 3)
ax.set_ylim(19.2, 10)
ax.set_xlabel(r"$G_{BP}-G_{RP}$")
ax.set_ylabel("G")
ax.legend()

plt.show()


In [ ]:
# Magnitude absoluta Gaia G (paralaxe em mas)
clean["M_G"] = clean.phot_g_mean_mag + 5 * np.log10(clean.parallax / 1000) + 5
cluster["M_G"] = cluster.phot_g_mean_mag + 5 * np.log10(cluster.parallax / 1000) + 5

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    clean.bp_rp,
    clean.M_G,
    s=3,
    alpha=0.03,
    label="Campo"
)

ax.scatter(
    cluster.bp_rp,
    cluster.M_G,
    s=9,
    alpha=0.6,
    label="Candidatos"
)

ax.set_xlim(-0.5, 3)
ax.set_ylim(15, -5)
ax.set_xlabel(r"$G_{BP}-G_{RP}$")
ax.set_ylabel(r"$M_G$")
ax.legend()

plt.show()

## Catálogo final

Salvamos apenas os objetos selecionados como candidatos do aglomerado. Este catálogo pode ser usado diretamente nos próximos exemplos, incluindo o crossmatch com DESI.


In [ ]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

cluster = cluster[
    [
        "source_id",
        "ra",
        "dec",
        "radius_deg",
        "pmra",
        "pmdec",
        "pmra_error",
        "pmdec_error",
        "parallax",
        "parallax_error",
        "phot_g_mean_mag",
        "phot_bp_mean_mag",
        "phot_rp_mean_mag",
        "bp_rp",
        "ruwe",
        "kinematic_excess",
    ]
].sort_values("kinematic_excess", ascending=False)

cluster.to_csv("outputs/cluster_1.csv", index=False)

print(f"{len(cluster):,} objetos salvos em outputs/cluster_1.csv")
cluster.head()
